# 2.5. Сравнительная оценка методов обработки данных

Сравниваются способы локальной загрузки, подготовки и объединения данных на одинаковых наборах строк. Для каждого способа измеряется время выполнения на текущем устройстве; указывается размер обработанных данных и проверяется сопоставимость результатов.

Дополнительно приводятся фактические времена этапов подготовки из 2.3 и анализа из 2.4. Время обучения Random Forest и его качество сравниваются с медианным базовым прогнозом.

Результаты времени применимы к текущему устройству и версиям библиотек. Сетевое скачивание источников в 2.2 отдельно не сравнивается: его длительность в сохранённом ноутбуке не записывалась.

In [6]:
from pathlib import Path
from statistics import median
import time

import duckdb
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from IPython.display import display

ROOT = next(
    (
        path
        for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "notebooks").is_dir() and (path / "README.md").exists()
    ),
    None,
)

if ROOT is None:
    raise FileNotFoundError("Не удалось найти корень проекта.")

RAWG_CSV = ROOT / "raw" / "rawg" / "videogames_data.csv"
RAWG_PARQUET = ROOT / "raw" / "rawg" / "videogames_data.parquet"
STEAM_PARQUET = ROOT / "processed" / "games" / "steam_games.parquet"
PLAYERS_PARQUET = ROOT / "processed" / "players" / "steam_monthly_players.parquet"
QUALITY_DIR = ROOT / "docs" / "data_quality"

TIME_23 = QUALITY_DIR / "processing_times_2_3.csv"
TIME_24 = QUALITY_DIR / "processing_times_2_4.csv"
METRICS_24 = QUALITY_DIR / "model_metrics_2_4.csv"

required_paths = [
    RAWG_CSV, RAWG_PARQUET, STEAM_PARQUET, PLAYERS_PARQUET,
    TIME_23, TIME_24, METRICS_24,
]
missing_paths = [path for path in required_paths if not path.is_file()]

if missing_paths:
    raise FileNotFoundError(
        "Сначала запусти 2.2–2.4. Не найдены файлы:\n"
        + "\n".join(str(path) for path in missing_paths)
    )


def benchmark(function, repeats=3):
    """Разогрев и медиана времени нескольких одинаковых запусков."""
    function()

    durations = []
    result = None

    for _ in range(repeats):
        started = time.perf_counter()
        result = function()
        durations.append(time.perf_counter() - started)

    return median(durations), result


print("Корень проекта:", ROOT)
print("Число повторов каждого замера: 3")

Корень проекта: C:\Users\Daniil\Documents\GitHub\gaming-data-analysis
Число повторов каждого замера: 3


In [7]:
BENCH_ROWS = 100_000
RAWG_COLUMNS = ["id", "name", "released", "rating", "ratings_count"]


def read_rawg_csv():
    return pd.read_csv(
        RAWG_CSV,
        usecols=RAWG_COLUMNS,
        nrows=BENCH_ROWS,
    )


def read_rawg_parquet():
    batches = []
    rows_read = 0

    for batch in pq.ParquetFile(RAWG_PARQUET).iter_batches(
        batch_size=BENCH_ROWS,
        columns=RAWG_COLUMNS,
    ):
        batches.append(batch)
        rows_read += batch.num_rows
        if rows_read >= BENCH_ROWS:
            break

    return pa.Table.from_batches(batches).slice(0, BENCH_ROWS).to_pandas()


csv_load_seconds, rawg_csv_sample = benchmark(read_rawg_csv)
parquet_load_seconds, rawg_sample = benchmark(read_rawg_parquet)

if len(rawg_csv_sample) != len(rawg_sample):
    raise ValueError("CSV и Parquet вернули разное число строк.")

load_results = pd.DataFrame([
    {
        "stage": "Загрузка RAWG, первые 100 000 строк",
        "method": "Pandas: CSV",
        "rows": len(rawg_csv_sample),
        "seconds": csv_load_seconds,
    },
    {
        "stage": "Загрузка RAWG, первые 100 000 строк",
        "method": "PyArrow: Parquet",
        "rows": len(rawg_sample),
        "seconds": parquet_load_seconds,
    },
])

print(f"Размер CSV: {RAWG_CSV.stat().st_size / 1024**2:.1f} МБ")
print(f"Размер Parquet: {RAWG_PARQUET.stat().st_size / 1024**2:.1f} МБ")
display(load_results)

# Подготовка одних и тех же уже загруженных строк в Pandas и DuckDB.
def prepare_rawg_pandas():
    result = rawg_sample[
        ["id", "released", "rating", "ratings_count"]
    ].copy()

    year = result["released"].astype("string").str.extract(
        r"^([0-9]{4})", expand=False
    )
    result["release_year"] = pd.to_numeric(year, errors="coerce")
    result["rating"] = pd.to_numeric(result["rating"], errors="coerce")
    result["ratings_count"] = pd.to_numeric(
        result["ratings_count"], errors="coerce"
    )

    return result[["id", "release_year", "rating", "ratings_count"]]


prep_connection = duckdb.connect(":memory:")
prep_connection.register("rawg_sample", rawg_sample)


def prepare_rawg_duckdb():
    return prep_connection.execute("""
        SELECT
            id,
            TRY_CAST(
                REGEXP_EXTRACT(
                    CAST(released AS VARCHAR),
                    '^([0-9]{4})',
                    1
                ) AS INTEGER
            ) AS release_year,
            TRY_CAST(rating AS DOUBLE) AS rating,
            TRY_CAST(ratings_count AS DOUBLE) AS ratings_count
        FROM rawg_sample
    """).fetchdf()


pandas_prep_seconds, prepared_pandas = benchmark(prepare_rawg_pandas)
duckdb_prep_seconds, prepared_duckdb = benchmark(prepare_rawg_duckdb)

if not prepared_pandas["id"].astype("string").equals(
    prepared_duckdb["id"].astype("string")
):
    raise ValueError("Pandas и DuckDB вернули разные игры.")

for column in ["release_year", "rating", "ratings_count"]:
    pandas_values = pd.to_numeric(
        prepared_pandas[column], errors="coerce"
    ).to_numpy(dtype=float)
    duckdb_values = pd.to_numeric(
        prepared_duckdb[column], errors="coerce"
    ).to_numpy(dtype=float)

    if not np.allclose(
        pandas_values, duckdb_values, equal_nan=True, rtol=1e-9, atol=1e-9
    ):
        raise ValueError(f"Результаты подготовки не совпали: {column}")

preparation_results = pd.DataFrame([
    {
        "stage": "Подготовка RAWG, 100 000 строк",
        "method": "Pandas",
        "rows": len(prepared_pandas),
        "seconds": pandas_prep_seconds,
    },
    {
        "stage": "Подготовка RAWG, 100 000 строк",
        "method": "DuckDB",
        "rows": len(prepared_duckdb),
        "seconds": duckdb_prep_seconds,
    },
])

display(preparation_results)
prep_connection.close()

Размер CSV: 1907.8 МБ
Размер Parquet: 964.8 МБ


,stage,method,rows,seconds
0,"Загрузка RAWG, первые 100 000 строк",Pandas: CSV,100000,1.607311
1,"Загрузка RAWG, первые 100 000 строк",PyArrow: Parquet,100000,0.012288


,stage,method,rows,seconds
0,"Подготовка RAWG, 100 000 строк",Pandas,100000,0.070137
1,"Подготовка RAWG, 100 000 строк",DuckDB,100000,0.010869


In [8]:
steam_keys = pd.read_parquet(
    STEAM_PARQUET,
    columns=["AppID"],
)
steam_keys["AppID"] = steam_keys["AppID"].astype("string")

players = pd.read_parquet(
    PLAYERS_PARQUET,
    columns=["steam_appid", "month", "avg_players"],
)

window_end = pd.Timestamp(players["month"].max())
window_start = window_end - pd.DateOffset(months=11)
recent_players = players[
    players["month"].between(window_start, window_end)
].copy()

def merge_pandas():
    recent = players[
        players["month"].between(window_start, window_end)
    ]

    player_summary = (
        recent.groupby("steam_appid", as_index=False)
        .agg(
            avg_players_12m=("avg_players", "mean"),
            months_observed=("month", "nunique"),
        )
    )
    player_summary["AppID"] = player_summary["steam_appid"].astype("string")

    return steam_keys.merge(
        player_summary[["AppID", "avg_players_12m", "months_observed"]],
        on="AppID",
        how="left",
        validate="one_to_one",
    )


merge_connection = duckdb.connect(":memory:")
merge_connection.register("steam_keys", steam_keys)
merge_connection.register("players", players)


def merge_duckdb():
    return merge_connection.execute(
        """
        WITH player_summary AS (
            SELECT
                CAST(steam_appid AS VARCHAR) AS AppID,
                AVG(avg_players) AS avg_players_12m,
                COUNT(DISTINCT month) AS months_observed
            FROM players
            WHERE month BETWEEN ? AND ?
            GROUP BY steam_appid
        )
        SELECT
            s.AppID,
            p.avg_players_12m,
            p.months_observed
        FROM steam_keys AS s
        LEFT JOIN player_summary AS p USING (AppID)
        """,
        [window_start.to_pydatetime(), window_end.to_pydatetime()],
    ).fetchdf()


pandas_merge_seconds, merged_pandas = benchmark(merge_pandas)
duckdb_merge_seconds, merged_duckdb = benchmark(merge_duckdb)

if len(merged_pandas) != len(merged_duckdb):
    raise ValueError("Pandas и DuckDB вернули разное число строк.")

left = merged_pandas.sort_values("AppID").reset_index(drop=True)
right = merged_duckdb.sort_values("AppID").reset_index(drop=True)

if not left["AppID"].astype("string").equals(
    right["AppID"].astype("string")
):
    raise ValueError("Pandas и DuckDB связали разные AppID.")

if not np.allclose(
    left["avg_players_12m"].to_numpy(dtype=float),
    right["avg_players_12m"].to_numpy(dtype=float),
    equal_nan=True,
    rtol=1e-9,
    atol=1e-9,
):
    raise ValueError("Средние значения игроков не совпали.")

merge_results = pd.DataFrame([
    {
        "stage": "Агрегация игроков и объединение со Steam",
        "method": "Pandas groupby + merge",
        "rows": len(merged_pandas),
        "matched_games": int(merged_pandas["avg_players_12m"].notna().sum()),
        "seconds": pandas_merge_seconds,
    },
    {
        "stage": "Агрегация игроков и объединение со Steam",
        "method": "DuckDB SQL",
        "rows": len(merged_duckdb),
        "matched_games": int(merged_duckdb["avg_players_12m"].notna().sum()),
        "seconds": duckdb_merge_seconds,
    },
])

print("Окно данных:", window_start.date(), "—", window_end.date())
display(merge_results)
merge_connection.close()

Окно данных: 2024-10-01 — 2025-09-01


,stage,method,rows,matched_games,seconds
0,Агрегация игроков и объединение со Steam,Pandas groupby + merge,150466,4336,0.098894
1,Агрегация игроков и объединение со Steam,DuckDB SQL,150466,4336,0.028305


In [9]:
times_23 = pd.read_csv(TIME_23)
times_24 = pd.read_csv(TIME_24)
metrics_24 = pd.read_csv(METRICS_24)

project_times = []

for row in times_23.itertuples(index=False):
    project_times.append({
        "stage": f"2.3: {row.stage}",
        "method": "Фактический замер этапа",
        "rows": pd.NA,
        "seconds": row.seconds,
    })

project_times.append({
    "stage": "2.4: весь анализ",
    "method": "Фактический замер этапа",
    "rows": pd.NA,
    "seconds": times_24.iloc[0]["seconds"],
})

for row in metrics_24.itertuples(index=False):
    project_times.append({
        "stage": f"2.4: Random Forest — {row.target}",
        "method": "Обучение и прогноз",
        "rows": row.sample_rows,
        "seconds": row.training_seconds,
    })

processing_results = pd.DataFrame(project_times)

method_comparison = pd.concat(
    [
        load_results,
        preparation_results,
        merge_results,
        processing_results,
    ],
    ignore_index=True,
)

method_comparison.to_csv(
    QUALITY_DIR / "method_comparison_2_5.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Сравнение этапов обработки")
display(method_comparison)

print("Качество моделей из 2.4: Random Forest против медианного прогноза")
display(
    metrics_24[
        [
            "target", "sample_rows", "model_mae", "baseline_mae",
            "model_rmse", "baseline_rmse", "model_r2", "training_seconds",
        ]
    ]
)

print(
    "Ускорение чтения Parquet относительно CSV:",
    f"{csv_load_seconds / parquet_load_seconds:.2f}×",
)
print(
    "Ускорение объединения DuckDB относительно Pandas:",
    f"{pandas_merge_seconds / duckdb_merge_seconds:.2f}×",
)

Сравнение этапов обработки


,stage,method,rows,seconds,matched_games
0,"Загрузка RAWG, первые 100 000 строк",Pandas: CSV,100000,1.607311,NaN
1,"Загрузка RAWG, первые 100 000 строк",PyArrow: Parquet,100000,0.012288,NaN
2,"Подготовка RAWG, 100 000 строк",Pandas,100000,0.070137,NaN
3,"Подготовка RAWG, 100 000 строк",DuckDB,100000,0.010869,NaN
4,Агрегация игроков и объединение со Steam,Pandas groupby + merge,150466,0.098894,4336.0
5,Агрегация игроков и объединение со Steam,DuckDB SQL,150466,0.028305,4336.0
6,2.3: raw_inventory,Фактический замер этапа,<NA>,3.791722,NaN
7,2.3: prepare_steam_rawg,Фактический замер этапа,<NA>,6.434671,NaN
8,2.3: prepare_csv_twitch_youtube_metadata,Фактический замер этапа,<NA>,3.903068,NaN
9,2.3: prepare_wikidata_habr,Фактический замер этапа,<NA>,0.518738,NaN


Качество моделей из 2.4: Random Forest против медианного прогноза


,target,sample_rows,model_mae,baseline_mae,model_rmse,baseline_rmse,model_r2,training_seconds
0,recommendations,142029,1.272159,0.984254,2.087538,2.581167,0.234621,2.551428
1,positive_share,82970,0.175486,0.178067,0.230375,0.244727,0.059354,1.157696
2,avg_players_12m,3551,1.414098,1.496284,1.860854,2.072658,0.139563,0.090512


Ускорение чтения Parquet относительно CSV: 130.80×
Ускорение объединения DuckDB относительно Pandas: 3.49×


## Выводы по сравнению методов

На текущем устройстве чтение первых 100 000 строк RAWG заняло 1,607 с для CSV и 0,012 с для Parquet. В этом замере Parquet прочитался примерно в 131 раз быстрее. Подготовка одинаковой выборки заняла 0,070 с в Pandas и 0,011 с в DuckDB; результаты методов совпали.

Объединение каталога Steam с месячной статистикой игроков заняло 0,099 с в Pandas и 0,028 с в DuckDB. Оба метода вернули 150 466 строк и данные об игроках для 4 336 игр.

В 2.3 отдельно измерены инвентаризация (3,792 с), подготовка Steam и RAWG (6,435 с), подготовка остальных табличных данных (3,903 с), подготовка Wikidata и Habr (0,519 с), NLP (1,610 с), сопоставление источников (4,531 с) и запись в DuckDB (3,877 с). Полный анализ 2.4 занял 12,786 с.

Random Forest обучался 2,551 с для рекомендаций, 1,158 с для доли положительных оценок и 0,091 с для числа игроков. Для рекомендаций результат смешанный: RMSE модели ниже ошибки медианного базового прогноза (2,088 против 2,581), но MAE выше (1,272 против 0,984); R² = 0,235. Для доли положительных оценок и числа игроков преимущество над базовым прогнозом по MAE и RMSE небольшое, а R² остаётся низким. Предсказательная способность используемых признаков ограничена.

Замеры относятся к текущему устройству, библиотекам и состоянию файлового кэша. Сетевое получение данных в 2.2 пока не замеряется: в этом ноутбуке нет сохранения времени скачивания Kaggle- и API-источников. Локальное чтение CSV и Parquet измерено отдельно в 2.5.